In [ ]:
"""Flow-matching training of a transformer velocity field on pMSSM data.

Converted from tests/training.ipynb. Differences from the notebook:
  * training batches come from the normalized training split, shuffled each epoch
  * validation uses the whole validation split with fixed noise/times/masks
  * AdamW with linear warmup + cosine decay, gradient clipping

Outputs (prefix = <out_dir>/<run_name>/<run_name>, out_dir defaults to tests/checkpoints):
  <prefix>_weights.pt       dict with model_state, hparams, mean, std, file_names, ...
  <prefix>_loss_history.npz arrays: step, train_loss, val_loss, lr
  <prefix>_loss.png         train/val loss curve
"""
import argparse
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from flow_matching.path import AffineProbPath
from flow_matching.path.scheduler import CondOTScheduler

from v2_fm_pmssm.data import load_data, pMSSM_Dataset, select, all_var, split_indices
from v2_fm_pmssm.training import Encoder_Transformer, make_masks, EpochSampler, fm_loss, evaluate, lr_at

In [ ]:
hparams = {
    "data_folder": "/u/yinya/archive/gen_flat_v2/ntuples",
    "max_files": None,
    "out_dir": "checkpoints",   # tests/checkpoints (the notebook runs from tests/)
    "run_name": f"fm_{os.environ.get('SLURM_JOB_ID', 'local')}",
    "batch_size": 1024,
    "learning_rate": 3e-4,
    "min_lr": 1e-5,
    "weight_decay": 0.01,
    "max_iters": 100_000,
    "warmup_frac": 0.02,        # fraction of max_iters
    "warmup_iters": None,       # optional absolute override of warmup_frac
    "eval_interval": 1_000,
    "eval_batch_size": 8192,
    "grad_clip": 1.0,
    "n_embd": 128,
    "n_heads": 4,
    "n_layers": 6,
    "seed": 42,
}

In [ ]:
args = argparse.Namespace(**hparams)
if args.warmup_iters is None:
    args.warmup_iters = int(args.warmup_frac * args.max_iters)
if args.warmup_iters >= args.max_iters:
    raise ValueError(f"warmup_iters ({args.warmup_iters}) must be smaller than max_iters ({args.max_iters})")
hparams = vars(args)
print(json.dumps(hparams, indent=2))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
torch.manual_seed(args.seed)
g = torch.Generator(device=device).manual_seed(args.seed)

In [ ]:
# ---- data ----------------------------------------------------------------
_, parameter_table, point_id, file_names = load_data(args.data_folder, max_files=args.max_files)
dataset = pMSSM_Dataset(parameter_table, point_id, select(all_var), device)
dataset = dataset.filter_invalid(branches=["MO_Omega"])
print(f"valid samples: {len(dataset)}")
train_idx, val_idx, _ = split_indices(len(dataset), (0.8, 0.1, 0.1), seed=args.seed)
train, val = dataset.subset(train_idx), dataset.subset(val_idx)
del dataset
mean, std = train.transform()
val.transform(mean, std)
print(f"train: {len(train)}, val: {len(val)}")

masks = make_masks(device)
n_val = len(val)
val_set = (
    val.parameters,
    torch.randint(len(masks), (n_val,), device=device, generator=g),
    torch.rand(n_val, device=device, generator=g),
    torch.randn(n_val, 13, device=device, generator=g),
)

In [ ]:
# ---- model / optim -------------------------------------------------------
model = Encoder_Transformer(args.n_embd, args.n_layers, args.n_heads).to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
opt = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)
path = AffineProbPath(scheduler=CondOTScheduler())
sampler = EpochSampler(train.parameters, args.batch_size, g)

In [ ]:
# ---- train ---------------------------------------------------------------
history = {"step": [], "train_loss": [], "val_loss": [], "lr": []}
running, n_running = 0.0, 0
best_val, best_state = float("inf"), None
t0 = time.time()
model.train()
for step in range(args.max_iters):
    lr = lr_at(step, args.max_iters, args.warmup_iters, args.learning_rate, args.min_lr)
    for group in opt.param_groups:
        group["lr"] = lr

    target = sampler.next()
    B = target.shape[0]
    loss = fm_loss(model, path, masks, target,
                    torch.randint(len(masks), (B,), device=device, generator=g),
                    torch.rand(B, device=device, generator=g),
                    torch.randn(B, 13, device=device, generator=g))
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
    opt.step()
    running += loss.item()
    n_running += 1

    if (step + 1) % args.eval_interval == 0 or step == args.max_iters - 1:
        val_loss = evaluate(model, path, masks, val_set, args.eval_batch_size)
        train_loss = running / n_running
        running, n_running = 0.0, 0
        for k, v in zip(history, (step + 1, train_loss, val_loss, lr)):
            history[k].append(v)
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"step {step + 1}/{args.max_iters} epoch {sampler.epoch}: "
                f"train {train_loss:.4f}  val {val_loss:.4f}  lr {lr:.2e}  "
                f"{time.time() - t0:.0f}s", flush=True)


In [ ]:

# ---- save ----------------------------------------------------------------
run_dir = os.path.join(args.out_dir, args.run_name)  # one folder per run: <out_dir>/<run_name>/<run_name>_*
os.makedirs(run_dir, exist_ok=True)
prefix = os.path.join(run_dir, args.run_name)
torch.save({
    "model_state": {k: v.cpu() for k, v in model.state_dict().items()},  # final
    "best_model_state": best_state,                                     # lowest val loss
    "best_val_loss": best_val,
    "hparams": hparams,
    "mean": mean.cpu(), "std": std.cpu(),
    "variables": [v.branch for v in select(all_var)],
    "masks": masks.cpu(),
    "file_names": [str(f) for f in file_names],
}, f"{prefix}_weights.pt")
np.savez(f"{prefix}_loss_history.npz", **{k: np.asarray(v) for k, v in history.items()})


In [ ]:

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(history["step"], history["train_loss"], label="train")
ax.plot(history["step"], history["val_loss"], label="val")
ax.set_xlabel("step")
ax.set_ylabel("flow-matching loss")
ax.set_yscale("log")
ax.set_title(f"{args.run_name}  (best val {best_val:.4f})")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f"{prefix}_loss.png", dpi=150)
print(f"saved {prefix}_weights.pt, {prefix}_loss_history.npz, {prefix}_loss.png")